# Vignette 04: Custom Components

Demonstrates three extension points: custom preprocessors, custom models, and custom evaluation metrics.

*Runtime: ~1 minute on a modern laptop.*

**Dataset**: TCGA BRCA (Cancer Genome Atlas Network 2012; 1071 samples, 7 features)

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

from mlsurv import SurvivalLearner
from mlsurv.utils import load_survival_data
from mlsurv.models.base import BaseSurvivalModel
from mlsurv.preprocessing import TopKCoxSelector, VIFSelector

# Results are written to ./output/
os.makedirs("output", exist_ok=True)

## Load TCGA BRCA dataset

In [ ]:
X, y = load_survival_data(
    'data/tcga_brca.csv',
    time_col='time',
    event_col='event',
)
print(f"TCGA BRCA: n={len(X)}, features={X.shape[1]}, events={y['event'].sum()}")
X.head()

## Section 1: Custom preprocessor

Pass custom sklearn transformers to `setup()` for each preprocessing stage. Here we use KNN imputation, a RobustScaler+PCA pipeline as scaler, and a chained VIF + TopK Cox feature selector.

In [ ]:
learner = SurvivalLearner(X, y)
learner.setup(
    test_fraction=0.3,
    cv_folds=5,
    cv_repeats=3,
    random_state=42,
    imputer=KNNImputer(n_neighbors=7),
    scaler=Pipeline([
        ('robust', RobustScaler()),
        ('pca', PCA(n_components=0.95)),
    ]),
    feature_selector=Pipeline([
        ('vif', VIFSelector(threshold=10.0)),
        ('topk', TopKCoxSelector(k=6)),
    ]),
    n_jobs=-1,
)

# Baseline with default RSF
baseline = learner.train('rsf')
print(f"RSF with custom preprocessing - CV C-index: {baseline.cv_metric_mean:.3f}")

## Section 2: Custom model

Implement `BaseSurvivalModel` to integrate any survival estimator into the mlsurv pipeline. Required: `name`, `display_name`, `color`, `fit(X, y)`, `predict(X)`. Store the fitted inner estimator as `self.model_` to enable survival-function prediction and other base-class defaults.

To make a custom model **tunable**, add an optional `get_search_space()` hook returning an Optuna search space (same `(min, max[, "log"])` / `[choices]` format as the built-in models). The model is then tunable via `learner.train(model, tune=True)` — see the tuning cell below.

In [ ]:
class RegularizedCoxModel(BaseSurvivalModel):
    """Ridge-penalized Cox PH model (tunable on `alpha`)."""
    name = 'reg_cox'
    display_name = 'Regularized Cox'
    color = '#17becf'
    supports_random_state = False

    def __init__(self, alpha=0.1):
        self.alpha = alpha

    def fit(self, X, y):
        self.model_ = CoxPHSurvivalAnalysis(alpha=self.alpha)
        self.model_.fit(X, y)
        self.is_fitted_ = True
        self.n_features_in_ = X.shape[1]
        if hasattr(X, 'columns'):
            self.feature_names_in_ = X.columns.to_numpy()
        return self

    def predict(self, X):
        self._check_is_fitted()
        return self.model_.predict(X)

    def get_params(self, deep=True):
        return {'alpha': self.alpha}

    def set_params(self, **params):
        if 'alpha' in params:
            self.alpha = params['alpha']
        return self

    def get_search_space(self):
        """Declare a search space so this model is tunable via train(tune=True)."""
        return {'alpha': (1e-3, 10.0, 'log')}

print('RegularizedCoxModel defined.')

In [ ]:
# Register and train the custom model
learner2 = SurvivalLearner(X, y)
learner2.setup(test_fraction=0.3, cv_folds=5, cv_repeats=3, random_state=42, n_jobs=-1)

# Train default RSF as baseline
rsf_result = learner2.train('rsf')
print(f"RSF CV C-index: {rsf_result.cv_metric_mean:.3f}")

# Train custom model
custom_model = RegularizedCoxModel(alpha=0.1)
custom_result = learner2.train(custom_model)
print(f"Regularized Cox CV C-index: {custom_result.cv_metric_mean:.3f}")

### Tuning the custom model

Because `RegularizedCoxModel` declares a `get_search_space()`, it can be tuned just like a built-in model. Optuna clones the instance each trial and sets the suggested `alpha`. You can also override or supply the space per call with `search_spaces={'model': {...}}` (the per-call dict wins over the declared one).

**Limitations.** Custom-instance tuning is programmatic only: it is not available through the CLI (which accepts model *names*, not instances), and a custom instance cannot be passed in a multi-model `run([...])` list. The declared search space is used as-is, so the events-per-variable (EPV) auto-narrowing applied to built-in tree/neural models does not constrain custom spaces. Size your ranges to the dataset yourself on small cohorts.

In [ ]:
# Tune the custom model on its declared search space (small n_trials for speed)
tuned_custom = learner2.train(
    RegularizedCoxModel(), tune=True, n_trials=10, verbose=False,
)
print(f"Tuned Regularized Cox - CV C-index: {tuned_custom.cv_metric_mean:.3f}")
print(f"Best params: {tuned_custom.best_params_}")

## Section 3: Custom metric

Pass custom scorers to `evaluate()`. The signature is `scorer(estimator, X, y) -> float`.

**Note**: Time horizons must match the dataset's time units — TCGA BRCA uses months, so 1 year = 12, 3 years = 36.

In [ ]:
def truncated_c_index(estimator, X, y, time_horizon=60):
    """C-index restricted to subjects with events before time_horizon (months)."""
    mask = (y['time'] <= time_horizon) | (y['event'] == False)
    if mask.sum() < 10:
        return np.nan
    risk = estimator.predict(X[mask])
    return concordance_index_censored(
        y[mask]['event'], y[mask]['time'], risk
    )[0]


eval_result = learner2.evaluate(
    rsf_result,
    custom_metrics={
        'c_index_1yr': lambda m, X, y: truncated_c_index(m, X, y, 12),
        'c_index_3yr': lambda m, X, y: truncated_c_index(m, X, y, 36),
    },
)

print(f"Standard C-index:  {eval_result.c_index_test:.3f}")
c1yr = eval_result.concordance['c_index_1yr'].iloc[0]
c3yr = eval_result.concordance['c_index_3yr'].iloc[0]
print(f"1-year C-index:    {c1yr:.3f}" if not np.isnan(c1yr) else "1-year C-index:    N/A")
print(f"3-year C-index:    {c3yr:.3f}" if not np.isnan(c3yr) else "3-year C-index:    N/A")

## Combined workflow: custom preprocessor, model, and metric

In [ ]:
# Fresh learner with all custom components
learner3 = SurvivalLearner(X, y)
learner3.setup(
    test_fraction=0.3, cv_folds=5, cv_repeats=3, random_state=42,
    imputer=KNNImputer(n_neighbors=7),
    scaler=RobustScaler(),
    n_jobs=-1,
)

# Train default RSF and custom model
rsf = learner3.train('rsf')
reg_cox = learner3.train(RegularizedCoxModel(alpha=0.05))

# Evaluate both with custom metrics
custom_metrics = {
    'c_index_1yr': lambda m, X, y: truncated_c_index(m, X, y, 12),
}

print(f"{'Model':20s} {'CV C-index':>12s} {'Test C-index':>14s} {'1yr C-index':>14s}")
print('-' * 65)
for name, result in [('RSF', rsf), ('Regularized Cox', reg_cox)]:
    ev = learner3.evaluate(result, custom_metrics=custom_metrics, verbose=False)
    c1yr = ev.concordance['c_index_1yr'].iloc[0]
    c1yr_str = f"{c1yr:.3f}" if not np.isnan(c1yr) else 'N/A'
    print(f"{name:20s} {result.cv_metric_mean:12.3f} {ev.c_index_test:14.3f} {c1yr_str:>14s}")

## Generate report

In [ ]:
learner3.generate_report(
    'output/vig4_report.html',
    title='Custom Components Report',
)
print('Report saved to output/vig4_report.html')

## References

- Cancer Genome Atlas Network (2012). Comprehensive molecular portraits of human breast tumours. *Nature*, 490(7418):61–70. https://doi.org/10.1038/nature11412